# 🌊 Flood Depth Estimator — Retrain with Gemini Pro

This notebook labels your flood images using **Google Gemini Pro** (AI vision model), then trains the model.

**Time:** ~4-5 hours total (30 min setup + 30 min labeling + 2-3 hours training + 30 min deploy)

**Cost:** ~$0.01 for Gemini labels (free tier includes credits)

## Step 1: Install dependencies

In [ ]:
!pip install -q google-genai pillow pandas torch torchvision omegaconf pyyaml tqdm

## Step 2: Mount Drive & clone repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone --branch main https://github.com/Mishra-Kaumod/flood-depth-estimator.git /content/flood-depth-estimator
%cd /content/flood-depth-estimator
print("✅ Repo cloned")

## Step 3: Get Gemini API key

1. Go to https://ai.google.dev/
2. Click "Get API Key" → "Create API key in new project"
3. Copy the key and paste below

In [ ]:
import google.genai as genai
from google.colab import userdata

# Option 1: Get from Colab secrets (recommended)
try:
    api_key = userdata.get('GEMINI_API_KEY')
    print("✅ Using API key from Colab Secrets")
except:
    print("⚠️ No GEMINI_API_KEY in secrets. Add it: click 🔑 in left sidebar")
    api_key = input("Paste your Gemini API key here: ")

genai.configure(api_key=api_key)
print(f"✅ Gemini configured")

## Step 4: Upload flood images

In [ ]:
from google.colab import files
import os

print("📁 Select 50-200 flood photos...")
uploaded = files.upload()

print(f"\n✅ Uploaded {len(uploaded)} images")
for fn in list(uploaded.keys())[:3]:
    print(f"  • {fn}")
if len(uploaded) > 3:
    print(f"  ... and {len(uploaded)-3} more")

## Step 5: Label with Gemini Pro + Fallback to CV

In [ ]:
import base64, csv, sys
from PIL import Image
import numpy as np
from tqdm import tqdm

sys.path.insert(0, '.')
from src.reference_depth_estimator import ReferenceDepthEstimator

# Initialize CV fallback
cv_estimator = ReferenceDepthEstimator()

# Prepare to write CSV
rows = []
gemini_model = genai.GenerativeModel('gemini-1.5-flash')  # Fast and free

print("🔍 Labeling images with Gemini Pro...\n")

for filename in tqdm(sorted(uploaded.keys()), desc="Gemini labeling"):
    try:
        # Read image
        with open(filename, 'rb') as f:
            image_bytes = f.read()
        
        # Use new google.genai API
        response = genai.upload_file(filename)
        
        prompt = (
            "Analyze this FLOOD DAMAGE PHOTO. You are a flood depth assessment expert.\n\n"
            "Estimate the FLOOD WATER DEPTH in centimeters.\n\n"
            "Look for visual cues:\n"
            "• Vehicle submersion: tire only visible=30cm, bumper=45cm, door=90cm, roof=150cm\n"
            "• Person submersion: ankle=15cm, knee=45cm, waist=90cm, chest=125cm\n"
            "• Waterline on buildings/structures\n"
            "• Road curbs (~15cm height)\n\n"
            "Return ONLY a number with unit. Examples:\n"
            "25 cm\n"
            "45\n"
            "depth: 30 cm\n\n"
            "If no water visible, return 0.\n"
            "If water covers entire photo, return 150.\n\n"
            "RESPOND WITH ONLY THE NUMBER."
        )
        
        response_text = genai.GenerativeModel('gemini-1.5-flash').generate_content(
            [prompt, response]
        ).text
        
        # Parse depth from response
        depth_str = ''.join(c for c in response_text if c.isdigit() or c == '.')
        if depth_str:
            depth = float(depth_str.split('.')[0])  # take first number
            depth = max(0, min(200, depth))  # clamp to 0-200 cm
        else:
            raise ValueError(f"Could not parse depth: {response_text}")
        
        rows.append((filename, depth))
        
    except Exception as e:
        # Fallback to CV estimator
        try:
            img = Image.open(filename).convert('RGB')
            result = cv_estimator.estimate(np.array(img))
            depth = result['depth_cm']
            rows.append((filename, depth))
        except:
            print(f"⚠️  {filename}: skipped (error)")
            continue

print(f"\n✅ Labeled {len(rows)} images")

## Step 6: Save labels.csv and verify distribution

In [ ]:
import pandas as pd

# Save labels
with open('labels.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['filename', 'depth_cm'])
    w.writerows(rows)

df = pd.read_csv('labels.csv')
print("📊 DEPTH STATISTICS")
print("="*50)
print(df['depth_cm'].describe())
print("="*50)

# Verify no label collapse
zero_count = (df['depth_cm'] == 0).sum()
if zero_count > len(df) * 0.8:
    print(f"⚠️  WARNING: {zero_count}/{len(df)} images are 0 cm (no water)")
    print("   → Consider uploading images with more visible water")
else:
    print(f"✅ Good label distribution (only {zero_count} zero labels)")
    print(f"   Mean depth: {df['depth_cm'].mean():.1f} cm")
    print(f"   Std dev: {df['depth_cm'].std():.1f} cm")

## Step 7: Organize dataset (train/val split)

In [ ]:
import shutil

# Create folders
os.makedirs('flood_dataset/train', exist_ok=True)
os.makedirs('flood_dataset/val', exist_ok=True)

# 85% train, 15% val
split_idx = int(len(df) * 0.85)
train_df = df.iloc[:split_idx]
val_df = df.iloc[split_idx:]

# Copy files
for idx, row in train_df.iterrows():
    if os.path.exists(row['filename']):
        shutil.copy2(row['filename'], f'flood_dataset/train/{row["filename"]}')

for idx, row in val_df.iterrows():
    if os.path.exists(row['filename']):
        shutil.copy2(row['filename'], f'flood_dataset/val/{row["filename"]}')

# Save labels in dataset folder
shutil.copy2('labels.csv', 'flood_dataset/labels.csv')

print(f"✅ Dataset prepared:")
print(f"   Train: {len(train_df)} images")
print(f"   Val:   {len(val_df)} images")
print(f"\n📊 Depth range: {df['depth_cm'].min():.0f} - {df['depth_cm'].max():.0f} cm")

## Step 8: Verify labels load correctly

In [ ]:
from omegaconf import OmegaConf
from src.dataset import create_dataloaders

cfg = OmegaConf.load('config/config.yaml')
print("Config loaded. Testing dataloaders...")

try:
    train_loader, val_loader = create_dataloaders(cfg)
    print(f"✅ Train batches: {len(train_loader)}")
    print(f"✅ Val batches:   {len(val_loader)}")
    
    # Show first batch
    batch = next(iter(train_loader))
    images, depths = batch
    print(f"\n✅ First batch: {images.shape} → depths: {depths.shape}")
    print(f"   Depth range in batch: {depths.min():.2f} - {depths.max():.2f}")
except Exception as e:
    print(f"❌ Error: {e}")
    print("   Make sure flood_dataset/labels.csv exists")

## Step 9: TRAIN! (2-3 hours)

In [ ]:
!pip install -r requirements.txt -q

print("🚀 Starting training...\n")
!python -m src.train_water_aware \
  --config config/config.yaml \
  --epochs 30 \
  --batch-size 16 \
  --lr 0.001 \
  --device cuda

print("\n✅ Training complete!")

## Step 10: Check model quality

In [ ]:
import torch

ckpt = torch.load('models/best_flood_model_water_aware.pth', map_location='cpu', weights_only=False)
print("Model Checkpoint Info:")
print(f"  Epoch: {ckpt.get('epoch', '?')}")
print(f"  Best Val Loss: {ckpt.get('best_val_loss', '?')}")
print(f"  Training History: {len(ckpt.get('training_history', []))} epochs logged")

# Check for label collapse
val_loss = ckpt.get('best_val_loss', 1.0)
if isinstance(val_loss, float) and val_loss < 1e-5:
    print("\n⚠️  WARNING: Model shows label collapse (val_loss ≈ 0)")
    print("   → Retrain with better labeled data")
else:
    print(f"\n✅ Model trained successfully (val_loss = {val_loss:.6f})")
    print("   Ready to deploy!")

## Step 11: Download model

In [ ]:
from google.colab import files

print("📥 Downloading model (50MB)...")
files.download('models/best_flood_model_water_aware.pth')
print("✅ Downloaded to your computer!")
print("\nNext steps (on your local machine):")
print("1. Move file to: D:\\best_flood_model_water_aware.pth")
print("2. Copy to repo: models/best_flood_model_water_aware.pth")
print("3. Push to GitHub: git add models/ && git commit && git push")
print("4. Server auto-reloads at http://localhost:5000")